# This notebook first defines GEVs then smaples from them and    
# then uses Bayseian to calculate the percentiles and computes the bias
##### Author: Omid Emamjomehzadeh (https://www.omidemam.com/)
##### Supervisor: Dr. Omar Wani (https://engineering.nyu.edu/faculty/omar-wani)
##### Hydrologic Systems Group @NYU (https://www.omarwani.com/)

In [1]:
#import libraries
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib as mpl
import pandas as pd
import numpy as np
import pymc3 as pm3
import scipy.stats as stats
from scipy import stats
from scipy.stats import genextreme
from scipy.stats import norm, gaussian_kde
from scipy.optimize import minimize
import seaborn as sns
import arviz as az
import pymc3 as pm
import os
import theano.tensor as tt
from theano.compile.ops import as_op
import theano
import pandas as pd
from pymc3.distributions.dist_math import bound
from scipy.stats import genextreme
import warnings
from arviz.plots import plot_utils as azpu
import arviz as az
from tqdm import tqdm
%matplotlib inline
sns.set()
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore', UserWarning)
theano.config.warn.round=False
from watermark import watermark
from datetime import datetime
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
import matplotlib.pyplot as plt
plt.style.use('default')   # <-- resets to plain Matplotlib look
from itertools import chain
import random


WARNING (theano.tensor.blas): Using NumPy C-API based implementation for BLAS functions.
The version of PyMC you are using is very outdated.

Please upgrade to the latest version of PyMC https://www.pymc.io/projects/docs/en/stable/installation.html

Also notice that PyMC3 has been renamed to PyMC.


In [2]:
# read parameters
mle_1h = r"D:\BMM-IDF4Drainage_data_results\Percentile\gev_params_1h_mle_AORC.csv"
# Read the path
df_mle_1h = pd.read_csv(mle_1h)
df_mle_1h

,City,shape(c),loc,scale
0,Birmingham,0.072627,22.491624,10.228208
1,Phoenix,-0.267678,8.175461,3.664874
2,Little Rock,-0.055021,21.988082,9.258394
3,Los Angeles,0.042387,11.861270,5.693252
4,Denver,-0.413798,7.283690,4.342796
5,Hartford,-0.285448,14.298267,6.394461
6,Wilmington,-0.035867,18.485770,8.115728
7,Miami,-0.285025,19.471994,9.999978
8,Atlanta,-0.226416,16.665304,7.189982
9,Boise,-0.174572,5.606417,2.189396


In [2]:
import contextlib, os
import logging, warnings
# Quiet PyMC3/Theano/ArviZ info logs
for name in ("pymc3", "pymc", "theano", "aesara", "pytensor", "arviz"):
    logging.getLogger(name).setLevel(logging.ERROR)

# Turn off the progress bar from pm.sample
PM_PROGRESSBAR = dict(progressbar=False)

# Hide the specific divergence warnings
warnings.filterwarnings(
    "ignore",
    message=r".*divergences? after tuning.*",   # matches "There was/There were ... divergences after tuning."
    category=UserWarning,
)
# (Optional) also hide other sampling/diagnostic warnings from PyMC/ArviZ
warnings.filterwarnings("ignore", category=UserWarning, module=r".*pymc3|pymc.*")
warnings.filterwarnings("ignore", category=UserWarning, module=r".*arviz.*")

In [3]:
# Likelihood Function
def gev_logp(value,μ  ,  σ,  ξ):
        scaled = (value - μ ) /  σ
        # non-zero
        logp_xi_not_zero = -(tt.log( σ)
                 + (( ξ + 1) /  ξ) * tt.log1p( ξ * scaled)
                 + (1 +  ξ * scaled) ** (-1/ ξ))
        #zero
        logp_xi_zero = -tt.log( σ) + ( ξ+1)*(-(value - μ )/ σ) - tt.exp(-(value - μ )/ σ)
        #combined
        logp = tt.switch(tt.abs_( ξ) > 1e-4  , logp_xi_not_zero, logp_xi_zero) 
        return tt.sum(logp)

In [4]:
def add_aorc_noise(values, start_year=1979, end_year=2024, seed=None):
    """
    Adds AORC-style multiplicative lognormal noise to rainfall values 
    for the CONUS domain, using year-based GM and GCV statistics.

    Parameters
    ----------
    values : np.ndarray
        1D array of rainfall values (first element = start_year, last = end_year).
    start_year : int, optional
        First year of the dataset. Default is 1979.
    end_year : int, optional
        Last year of the dataset. Default is 2024.
    seed : int or None, optional
        Random seed for reproducibility.

    Returns
    -------
    noisy_values : np.ndarray
        Array of rainfall values with multiplicative AORC noise applied.
    """

    # --- Check inputs ---
    n_years = end_year - start_year + 1
    if len(values) != n_years:
        raise ValueError(f"Expected {n_years} values (one per year), got {len(values)}.")

    # --- AORC CONUS GM and GCV table (Table A1) ---
    eras = [
        (1800, 2001, 0.98, 1.8),
        (2002, 2009, 0.98, 2.4),
        (2010, 2015, 1.00, 3.7),
        (2016, 2021, 1.00, 2.9),
        (2022, 2100, 1.00, 2.9),  # extrapolate last period
    ]

    rng = np.random.default_rng(seed)
    noisy_values = np.zeros_like(values, dtype=float)

    # --- Loop over each year and apply lognormal noise ---
    for i, year in enumerate(range(start_year, end_year + 1)):
        # Get GM and GCV for that year
        for y0, y1, GM, GCV in eras:
            if y0 <= year <= y1:
                mu = np.log(GM)
                sigma = np.sqrt(np.log(1 + (GCV / 100.0) ** 2))
                MB = rng.lognormal(mean=mu, sigma=sigma)  # multiplicative bias
                noisy_values[i] = values[i] * MB
                break

    return noisy_values

# NYC

In [7]:
# -*- coding: utf-8 -*-
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
from scipy.stats import genextreme
# import pymc as pm  # uncomment if not already imported elsewhere

# ---------------- Helpers ----------------
def unpack_betas(beta, x):
    """
    Given beta (length 6) and covariate x (n,):
      xi(x)      = 0.5*tanh(b0 + b1*x)        (shape, in (-0.5, 0.5))
      mu(x)      = b2 + b3*x                  (location)
      log(s)(x)  = b4 + b5*x  -> s(x)=exp(.)  (scale>0)
    Returns arrays c (shape), loc, scale (each length n).
    """
    b0, b1, b2, b3, b4, b5 = beta
    x = np.asarray(x, dtype=float)
    c     = 0.5 * np.tanh(b0 + b1*x)
    loc   = b2 + b3*x
    scale = np.exp(b4 + b5*x)
    return c, loc, scale

def params_from_beta_for_years(beta, years, first_year):
    """
    Map calendar 'years' to the time index x used in fitting.
    If the fit used x = year - first_year, then x_years = years - first_year.
    """
    years = np.asarray(years, dtype=float)
    x_years = years - float(first_year)
    return unpack_betas(beta, x_years)

# ---------------- Load fitted betas ----------------
beta_table = pd.read_csv(
    r"D:\BMM-IDF4Drainage_data_results\Percentile\gev_nonstationary_linear_betas.csv",
    index_col=0
)

# ----- SETTINGS (edit first_year if your fit alignment differs) -----
FIRST_YEAR_FIT = 1979             # calendar year corresponding to x=0 during fitting
AORC_END_YEAR  = 2024             # window ends here
SAMPLE_SIZE    = 46               # last N years
MC_SIZE_PER_YEAR = 10_000         # for pooled truth
N_SIMS         = 500

# Return periods and non-exceedance probs
T = np.array([2, 5, 10, 25, 50, 100], dtype=float)
p = 1.0 - 1.0 / T                 # equals percentiles [50,80,90,96,98,99]%

# Seeds
random.seed(12345)
seeds = random.sample(range(1, 100001), N_SIMS)

# ---------------- Work on 29th row only ----------------
row_29 = beta_table.iloc[29]
city   = row_29.name
b = row_29[[
    "xi_intercept", "xi_slope",
    "mu_intercept", "mu_slope",
    "logscale_intercept", "logscale_slope"
]].to_numpy(dtype=float)

# Year window (last N years ending AORC_END_YEAR)
end_year   = AORC_END_YEAR
start_year = end_year - SAMPLE_SIZE + 1
years_window = np.arange(start_year, end_year + 1, dtype=float)  # length = SAMPLE_SIZE

# Nonstationary parameters per year in the window
c_years, loc_years, scale_years = params_from_beta_for_years(
    b, years_window, first_year=FIRST_YEAR_FIT
)

# ---- TRUE return levels (pooled mixture over the last N years) ----
rng_for_true = np.random.default_rng(seeds[0])  # fixed per (city, window)
pooled_mc = genextreme.rvs(
    c_years[:, None], loc=loc_years[:, None], scale=scale_years[:, None],
    size=(SAMPLE_SIZE, MC_SIZE_PER_YEAR), random_state=rng_for_true
).ravel(order='C')
rl_true = np.quantile(pooled_mc, p)  # shape (6,)

# ---------------- Simulations + Bayesian fit ----------------
bias = np.zeros((N_SIMS, len(T)), dtype=float)

# Percentiles list consistent with p (for quick quantiles on simulated draws)
percentiles = (p * 100.0).tolist()  # [50, 80, 90, 96, 98, 99]

for k in range(1, N_SIMS + 1):  # 1-based loop
    try:
        rng = np.random.default_rng(seeds[k-1])

        # exactly one simulated observation per year using nonstationary params
        samples = genextreme.rvs(
            c=c_years, loc=loc_years, scale=scale_years, size=None, random_state=rng
        )

        # Add AORC noise year-by-year (assumed defined)
        obs_samples = add_aorc_noise(
            samples, start_year=start_year, end_year=end_year, seed=seeds[k-1]
        )

        # --------------- Bayesian model ---------------
        # Assumes `pm` and `gev_logp` are available in your environment.
        n_draws = 1000
        sample_number = 3500
        chain_n = 4

        # initialize the arrays
        save_samples_24h = np.zeros([3, sample_number * chain_n])
        p_bayes_all_cities = np.zeros([sample_number * chain_n, n_draws])
        percentiles_all_cities = np.zeros([len(T)])

        with pm.Model() as model_gev:
            m = np.median(obs_samples)
            μ = pm.Normal("μ", mu=m, sigma=100)
            σ = pm.Lognormal("σ", mu=4, sigma=1)
            ξ = pm.Normal("ξ", mu=1, sigma=1)

            gev_24h = pm.DensityDist('gev', gev_logp, observed={'value': obs_samples, 'μ': μ, 'σ': σ, 'ξ': ξ})
            step = pm.NUTS(target_accept=0.999, max_treedepth=15)
            trace_24h = pm.sample(
                sample_number, step=step, cores=4, chains=chain_n, tune=10_000,
                return_inferencedata=True, init="jitter+adapt_diag",
                idata_kwargs={"density_dist_obs": False}
            )

            stacked_24h = trace_24h.posterior.stack(draws=("chain", "draw"))

            shape_posterior1 = stacked_24h.ξ.values
            loc_posterior1   = stacked_24h.μ.values
            scale_posterior1 = stacked_24h.σ.values

            save_samples_24h[0, :] = shape_posterior1
            save_samples_24h[1, :] = loc_posterior1
            save_samples_24h[2, :] = scale_posterior1

            # SciPy genextreme uses 'c' as shape; if your ξ sign convention differs, adjust here.
            c_post   = (-save_samples_24h[0, :])[:, None]  # keep your original sign flip
            loc_post = ( save_samples_24h[1, :])[:, None]
            scale_post = (save_samples_24h[2, :])[:, None]

            # Draw 1000 samples for each posterior parameter set (3500*chains)
            draws = genextreme.rvs(
                c=c_post, loc=loc_post, scale=scale_post,
                size=(sample_number * chain_n, n_draws), random_state=12345
            )
            p_bayes_all_cities[:, :] = draws

            # Percentiles over all posterior predictive draws (vector of length 6)
            percentiles_all_cities[:] = np.percentile(draws.ravel(), percentiles)

        # Bias against nonstationary truth (rl_true already computed at same p)
        bias[k-1, :] = percentiles_all_cities[:] - rl_true

    except Exception:
                print(Exception)
                # if anything fails for this i, mark its bias as NaN and continue
                bias[k, :] = np.nan
                continue
    # -------- Save per-simulation bias series --------
    sim_idx = np.arange(1, N_SIMS + 1)
    df_bias_city = pd.DataFrame({'City': city, 'sim': sim_idx})
    for j, t in enumerate(T):
        df_bias_city[f'bias_T{int(t)}'] = bias[:, j]

    out_path = rf"D:\BMM-IDF4Drainage_data_results\Bias\Bayesian\bias_series\_nonstationary_bias_sim_series_{city}.csv"
    df_bias_city.to_csv(out_path, index=False)
    print(f"Saved: {out_path}")

The version of PyMC you are using is very outdated.

Please upgrade to the latest version of PyMC https://www.pymc.io/projects/docs/en/stable/installation.html

Also notice that PyMC3 has been renamed to PyMC.
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [ξ, σ, μ]


Sampling 4 chains for 10_000 tune and 3_500 draw iterations (40_000 + 14_000 draws total) took 135 seconds.
There were 10 divergences after tuning. Increase `target_accept` or reparameterize.
There were 4 divergences after tuning. Increase `target_accept` or reparameterize.
There were 11 divergences after tuning. Increase `target_accept` or reparameterize.
There were 4 divergences after tuning. Increase `target_accept` or reparameterize.


Saved: D:\BMM-IDF4Drainage_data_results\Bias\Bayesian\bias_series\_nonstationary_bias_sim_series_New York City.csv


The version of PyMC you are using is very outdated.

Please upgrade to the latest version of PyMC https://www.pymc.io/projects/docs/en/stable/installation.html

Also notice that PyMC3 has been renamed to PyMC.
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [ξ, σ, μ]


Sampling 4 chains for 10_000 tune and 3_500 draw iterations (40_000 + 14_000 draws total) took 154 seconds.
There were 2 divergences after tuning. Increase `target_accept` or reparameterize.
There were 3 divergences after tuning. Increase `target_accept` or reparameterize.
There was 1 divergence after tuning. Increase `target_accept` or reparameterize.
There were 3 divergences after tuning. Increase `target_accept` or reparameterize.


...

Sampling 4 chains for 10_000 tune and 3_500 draw iterations (40_000 + 14_000 draws total) took 126 seconds.
There were 3 divergences after tuning. Increase `target_accept` or reparameterize.
There were 3 divergences after tuning. Increase `target_accept` or reparameterize.


Saved: D:\BMM-IDF4Drainage_data_results\Bias\Bayesian\bias_series\_nonstationary_bias_sim_series_New York City.csv


The version of PyMC you are using is very outdated.

Please upgrade to the latest version of PyMC https://www.pymc.io/projects/docs/en/stable/installation.html

Also notice that PyMC3 has been renamed to PyMC.
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [ξ, σ, μ]


Sampling 4 chains for 10_000 tune and 3_500 draw iterations (40_000 + 14_000 draws total) took 142 seconds.
There were 2 divergences after tuning. Increase `target_accept` or reparameterize.
There were 3 divergences after tuning. Increase `target_accept` or reparameterize.
There were 2 divergences after tuning. Increase `target_accept` or reparameterize.
There was 1 divergence after tuning. Increase `target_accept` or reparameterize.


Saved: D:\BMM-IDF4Drainage_data_results\Bias\Bayesian\bias_series\_nonstationary_bias_sim_series_New York City.csv


# ALL cities

In [5]:
# ---------------- Helpers ----------------
def unpack_betas(beta, x):
    """
    Given beta (length 6) and covariate x (n,):
      xi(x)      = 0.5*tanh(b0 + b1*x)        (shape, in (-0.5, 0.5))
      mu(x)      = b2 + b3*x                  (location)
      log(s)(x)  = b4 + b5*x  -> s(x)=exp(.)  (scale>0)
    Returns arrays c (shape), loc, scale (each length n).
    """
    b0, b1, b2, b3, b4, b5 = beta
    x = np.asarray(x, dtype=float)
    c     = 0.5 * np.tanh(b0 + b1*x)
    loc   = b2 + b3*x
    scale = np.exp(b4 + b5*x)
    return c, loc, scale

def params_from_beta_for_years(beta, years, first_year):
    """
    Map calendar 'years' to the time index x used in fitting.
    If the fit used x = year - first_year, then x_years = years - first_year.
    """
    years = np.asarray(years, dtype=float)
    x_years = years - float(first_year)
    return unpack_betas(beta, x_years)

# ---------------- Load fitted betas ----------------
beta_table = pd.read_csv(
    r"D:\BMM-IDF4Drainage_data_results\Percentile\gev_nonstationary_linear_betas.csv",
    index_col=0
)

# ----- SETTINGS (edit first_year if your fit alignment differs) -----
FIRST_YEAR_FIT = 1979             # calendar year corresponding to x=0 during fitting
AORC_END_YEAR  = 2024             # window ends here
SAMPLE_SIZE    = 46               # last N years
MC_SIZE_PER_YEAR = 10_000         # for pooled truth
N_SIMS         = 500

# Return periods and non-exceedance probs
T = np.array([2, 5, 10, 25, 50, 100], dtype=float)
p = 1.0 - 1.0 / T                 # equals percentiles [50,80,90,96,98,99]%

# Seeds
random.seed(12345)
seeds = random.sample(range(1, 100001), N_SIMS)


for idx, row in beta_table.iterrows():

    row_i = beta_table.loc[idx]
    city   = row_i.name
    b = row_i[[
        "xi_intercept", "xi_slope",
        "mu_intercept", "mu_slope",
        "logscale_intercept", "logscale_slope"
    ]].to_numpy(dtype=float)

    # Year window (last N years ending AORC_END_YEAR)
    end_year   = AORC_END_YEAR
    start_year = end_year - SAMPLE_SIZE + 1
    years_window = np.arange(start_year, end_year + 1, dtype=float)  # length = SAMPLE_SIZE

    # Nonstationary parameters per year in the window
    c_years, loc_years, scale_years = params_from_beta_for_years(
        b, years_window, first_year=FIRST_YEAR_FIT
    )

    # ---- TRUE return levels (pooled mixture over the last N years) ----
    rng_for_true = np.random.default_rng(seeds[0])  # fixed per (city, window)
    pooled_mc = genextreme.rvs(
        c_years[:, None], loc=loc_years[:, None], scale=scale_years[:, None],
        size=(SAMPLE_SIZE, MC_SIZE_PER_YEAR), random_state=rng_for_true
    ).ravel(order='C')
    rl_true = np.quantile(pooled_mc, p)  # shape (6,)

    # ---------------- Simulations + Bayesian fit ----------------
    bias = np.zeros((N_SIMS, len(T)), dtype=float)

    # Percentiles list consistent with p (for quick quantiles on simulated draws)
    percentiles = (p * 100.0).tolist()  # [50, 80, 90, 96, 98, 99]

    for k in range(1, N_SIMS + 1):  # 1-based loop
        print (city+str(k))
        try:
            rng = np.random.default_rng(seeds[k-1])

            # exactly one simulated observation per year using nonstationary params
            samples = genextreme.rvs(
                c=c_years, loc=loc_years, scale=scale_years, size=None, random_state=rng
            )

            # Add AORC noise year-by-year (assumed defined)
            obs_samples = add_aorc_noise(
                samples, start_year=start_year, end_year=end_year, seed=seeds[k-1]
            )

            # --------------- Bayesian model ---------------
            # Assumes `pm` and `gev_logp` are available in your environment.
            n_draws = 1000
            sample_number = 3500
            chain_n = 4

            # initialize the arrays
            save_samples_24h = np.zeros([3, sample_number * chain_n])
            p_bayes_all_cities = np.zeros([sample_number * chain_n, n_draws])
            percentiles_all_cities = np.zeros([len(T)])

            with pm.Model() as model_gev:
                m = np.median(obs_samples)
                μ = pm.Normal("μ", mu=m, sigma=100)
                σ = pm.Lognormal("σ", mu=4, sigma=1)
                ξ = pm.Normal("ξ", mu=1, sigma=1)

                gev_24h = pm.DensityDist('gev', gev_logp, observed={'value': obs_samples, 'μ': μ, 'σ': σ, 'ξ': ξ})
                step = pm.NUTS(target_accept=0.999, max_treedepth=15)
                trace_24h = pm.sample(
                    sample_number, step=step, cores=4, chains=chain_n, tune=10_000,
                    return_inferencedata=True, init="jitter+adapt_diag",progressbar=False,
                    idata_kwargs={"density_dist_obs": False}
                )

                stacked_24h = trace_24h.posterior.stack(draws=("chain", "draw"))

                shape_posterior1 = stacked_24h.ξ.values
                loc_posterior1   = stacked_24h.μ.values
                scale_posterior1 = stacked_24h.σ.values

                save_samples_24h[0, :] = shape_posterior1
                save_samples_24h[1, :] = loc_posterior1
                save_samples_24h[2, :] = scale_posterior1

                # SciPy genextreme uses 'c' as shape; if your ξ sign convention differs, adjust here.
                c_post   = (-save_samples_24h[0, :])[:, None]  # keep your original sign flip
                loc_post = ( save_samples_24h[1, :])[:, None]
                scale_post = (save_samples_24h[2, :])[:, None]

                # Draw 1000 samples for each posterior parameter set (3500*chains)
                draws = genextreme.rvs(
                    c=c_post, loc=loc_post, scale=scale_post,
                    size=(sample_number * chain_n, n_draws), random_state=12345
                )
                p_bayes_all_cities[:, :] = draws

                # Percentiles over all posterior predictive draws (vector of length 6)
                percentiles_all_cities[:] = np.percentile(draws.ravel(), percentiles)

            # Bias against nonstationary truth (rl_true already computed at same p)
            bias[k-1, :] = percentiles_all_cities[:] - rl_true

        except Exception:
                    print(Exception)
                    # if anything fails for this i, mark its bias as NaN and continue
                    bias[k, :] = np.nan
                    continue
        # -------- Save per-simulation bias series --------
        sim_idx = np.arange(1, N_SIMS + 1)
        df_bias_city = pd.DataFrame({'City': city, 'sim': sim_idx})
        for j, t in enumerate(T):
            df_bias_city[f'bias_T{int(t)}'] = bias[:, j]

        out_path = rf"D:\BMM-IDF4Drainage_data_results\Bias\Bayesian\bias_series\Nonstationary\nonstationary_bias_sim_series_{city}.csv"
        df_bias_city.to_csv(out_path, index=False)
        print(f"Saved: {out_path}")

Birmingham1


There were 73 divergences after tuning. Increase `target_accept` or reparameterize.
There were 57 divergences after tuning. Increase `target_accept` or reparameterize.
There were 68 divergences after tuning. Increase `target_accept` or reparameterize.
There were 91 divergences after tuning. Increase `target_accept` or reparameterize.


Saved: D:\BMM-IDF4Drainage_data_results\Bias\Bayesian\bias_series\Nonstationary\nonstationary_bias_sim_series_Birmingham.csv
Birmingham2
Saved: D:\BMM-IDF4Drainage_data_results\Bias\Bayesian\bias_series\Nonstationary\nonstationary_bias_sim_series_Birmingham.csv
Birmingham3


There were 51 divergences after tuning. Increase `target_accept` or reparameterize.
There were 29 divergences after tuning. Increase `target_accept` or reparameterize.
There were 42 divergences after tuning. Increase `target_accept` or reparameterize.
There were 39 divergences after tuning. Increase `target_accept` or reparameterize.


Saved: D:\BMM-IDF4Drainage_data_results\Bias\Bayesian\bias_series\Nonstationary\nonstationary_bias_sim_series_Birmingham.csv
Birmingham4
Saved: D:\BMM-IDF4Drainage_data_results\Bias\Bayesian\bias_series\Nonstationary\nonstationary_bias_sim_series_Birmingham.csv
Birmingham5


...

There was 1 divergence after tuning. Increase `target_accept` or reparameterize.
There were 4 divergences after tuning. Increase `target_accept` or reparameterize.
There were 2 divergences after tuning. Increase `target_accept` or reparameterize.


Saved: D:\BMM-IDF4Drainage_data_results\Bias\Bayesian\bias_series\Nonstationary\nonstationary_bias_sim_series_Denver.csv
Denver496
<class 'Exception'>
Denver497
Saved: D:\BMM-IDF4Drainage_data_results\Bias\Bayesian\bias_series\Nonstationary\nonstationary_bias_sim_series_Denver.csv
Denver498
Saved: D:\BMM-IDF4Drainage_data_results\Bias\Bayesian\bias_series\Nonstationary\nonstationary_bias_sim_series_Denver.csv
Denver499


There were 2 divergences after tuning. Increase `target_accept` or reparameterize.
There was 1 divergence after tuning. Increase `target_accept` or reparameterize.


Saved: D:\BMM-IDF4Drainage_data_results\Bias\Bayesian\bias_series\Nonstationary\nonstationary_bias_sim_series_Denver.csv
Denver500
<class 'Exception'>


IndexError: index 500 is out of bounds for axis 0 with size 500

In [23]:
#Getting the current date and time
current_datetime = datetime.now()

# Printing the date and time
print("Date and Time of the Notebook Analysis:", current_datetime)

Date and Time of the Notebook Analysis: 2025-10-23 18:05:38.733360


In [24]:
%load_ext watermark

# Print the Python version and some dependencies
%watermark -v -m -p numpy,pandas,matplotlib,pymc3,scipy,seaborn,arviz,os,theano,warnings,tqdm,watermark


Python implementation: CPython
Python version       : 3.9.19
IPython version      : 8.18.1

numpy     : 1.22.1
pandas    : 2.0.3
matplotlib: 3.8.4
pymc3     : 3.11.6
scipy     : 1.7.3
seaborn   : 0.13.2
arviz     : 0.12.1
os        : unknown
theano    : 1.1.2
warnings  : unknown
tqdm      : 4.66.4
watermark : 2.4.3

Compiler    : MSC v.1929 64 bit (AMD64)
OS          : Windows
Release     : 10
Machine     : AMD64
Processor   : Intel64 Family 6 Model 183 Stepping 1, GenuineIntel
CPU cores   : 24
Architecture: 64bit

